In [3]:
from ultralytics import YOLO

# Load the model
model = YOLO('yolov8l.yaml')  # you can use other model variants like yolov8s.pt, yolov8m.pt, etc.


model.train(data='/Users/saudahmad/Desktop/yolov8/data.yaml',
            epochs=1,
            imgsz=640,
            batch=16, 
            project='results',
            name='weed_detection',
            exist_ok=True,
            save_period=1)  # Saves checkpoints every 5 epochs



PermissionError: [Errno 1] Operation not permitted

In [ ]:
import cv2
from ultralytics import YOLO


# Load the YOLOv8 model (replace 'best.pt' with the path to your trained model)
model = YOLO('/Users/saudahmad/Desktop/yolov8/scripts/results/weed_detection/weights/best.pt')

# Open a connection to the webcam
cap = cv2.VideoCapture(0)  # 0 is the default ID for the primary webcam

if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

while True:
    # Capture frame-by-frame
    ret, frame = cap.read()
    if not ret:
        print("Error: Could not read frame.")
        break

    # Use the model to make predictions on the frame
    results = model(frame)
    
    # Loop through the detections and draw bounding boxes and labels on the frame
    for result in results:
        boxes = result.boxes.xyxy.numpy()  # get box coordinates
        confidences = result.boxes.conf.numpy()  # get confidence scores
        classes = result.boxes.cls.numpy().astype(int)  # get class indices

        for box, confidence, cls in zip(boxes, confidences, classes):
            x1, y1, x2, y2 = box.astype(int)
            label = f'{model.names[cls]} {confidence:.2f}'
            
            # Draw rectangle and label
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    # Display the resulting frame
    cv2.imshow('Webcam - YOLOv8', frame)
    
    # Break the loop on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the webcam and close windows
cap.release()
cv2.destroyAllWindows()

In [ ]:
from inference_sdk import InferenceHTTPClient
import time

start_time = time.time()

CLIENT = InferenceHTTPClient(
    api_url="https://detect.roboflow.com",
    api_key="r1BvL5SHssBf78rJoCxf"
)

image_path = "/Users/saudahmad/Desktop/yolov8/simple_weeds/test/images/20210907_153931_x264_mp4-34_jpg.rf.379ce98bb5c004bc8bf2fdc22dbccf33.jpg"  # Replace with the actual path to your image
result = CLIENT.infer(image_path, model_id="weeds-nxe1w/1")

end_time = time.time()
elapsed_time = end_time - start_time

print(f"Elapsed time: {elapsed_time} seconds")

In [ ]:
from ultralytics import YOLO

model = YOLO('/Users/saudahmad/Desktop/yolov8/simple_weeds/yolov8n.pt')  # Load your trained model
model.export(format='onnx')  # Export to ONNX